# Model Testing Notebook

This notebook contains the logic for testing the trained model.

In [ ]:
import os
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Add project root to path for helpers
project_root = Path(os.path.abspath(os.path.join(os.getcwd(), '../../..')))
sys.path.append(str(project_root))

from src.utils.data_manager import load_from, save_to

# Scikit-Learn Imports
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, KBinsDiscretizer, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
print(f"Project Root: {project_root}")

In [ ]:
# Load Test Data
test_data_path = project_root / 'data' / 'model' / 'car_sales_data_cleaned_test.csv'
df_test = pd.read_csv(test_data_path)

y_test = df_test['Price']
X_test = df_test.drop(columns=['Price'])

print(f"Loaded Test Data: {X_test.shape}")

# Load Models  
model_dir = project_root / 'data' / 'model'
fitted_models = {}
fitted_models['Ensemble'] = joblib.load(model_dir / 'ensemble_model_best.pkl')
fitted_models['HistGradientBoosting_Tuned'] = joblib.load(model_dir / 'histgradientboosting_tuned_model.pkl')

print("Models loaded")

In [ ]:
def evaluate_model(name, pipeline, X_val, y_val):
    # Predict
    y_pred_log = pipeline.predict(X_val)
    
    # Invert Log Transform (Exp) to get Real Prices
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_val)
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'y_pred': y_pred, 'y_true': y_true}

# Evaluate Ensemble
metrics = evaluate_model('Ensemble', fitted_models['Ensemble'], X_test, y_test)
print("\nEnsemble Test Metrics:")
print(f"RMSE: ${metrics['RMSE']:,.2f}")
print(f"MAE: ${metrics['MAE']:,.2f}")
print(f"R²: {metrics['R2']:.4f}")

In [ ]:
# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Actual vs Predicted
sns.scatterplot(x=metrics['y_true'], y=metrics['y_pred'], alpha=0.5, ax=axes[0])
axes[0].plot([metrics['y_true'].min(), metrics['y_true'].max()], 
             [metrics['y_true'].min(), metrics['y_true'].max()], 
             'r--', lw=2)
axes[0].set_title('Actual vs Predicted Price')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')

# 2. Residuals
residuals = metrics['y_true'] - metrics['y_pred']
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Error ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Permutation Importance (Test Set)
print("Calculating Feature Importance...")
result = permutation_importance(
    fitted_models['Ensemble'], X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(10, 6))
plt.boxplot(
    result.importances[sorted_idx].T,
    vert=False,
    labels=X_test.columns[sorted_idx]
)
plt.title("Permutation Importances (Test Set)")
plt.tight_layout()
plt.show()

In [ ]:
# PDP for top numeric features using HistGradientBoosting (Directly supported)
print("Generating Partial Dependence Plots...")
common_features = ['Age', 'Mileage', 'Engine size']
Display = PartialDependenceDisplay.from_estimator(
    fitted_models['HistGradientBoosting_Tuned'], 
    X_test, 
    common_features,
    grid_resolution=20
)
plt.suptitle('Partial Dependence of Price on Key Features')
plt.subplots_adjust(top=0.9)
plt.show()